# Preprocess masks generated with HistoKit

This notebook includes the code to preprocess annotated regions from artifact masks generated with HistoKit.

## TCGA - Compass NMD Dataset

Annotations for the TCGA - Compass NMD dataset are available for regions on the WSI, so there is a need to crop the masks to the annotated regions.

Masks are cropped using the coordinates saved in the file `coords.csv` and saved in a new folder. The cropped masks are also saved in color format for visualization purposes.

In [ ]:
from PIL import Image
import numpy as np
from histokit.savers import HDF5Saver
import pandas as pd
import os
import shutil
from tqdm import tqdm
from histokit.file_utils.file_check import check_gt_pred_folders
import os
Image.MAX_IMAGE_PIXELS = None

classes = {
    "Tissue": [128, 128, 128],
    "Background": [0, 0, 0],
    "Fold": [255, 99, 71],
    "Dark.Spot": [0, 255, 0],
    "Pen": [255, 0, 0],
    "Edge": [255, 0, 255],
    "Out.Of.Focus": [75, 0, 130],
}



In [ ]:
df = pd.read_csv('/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/TCGA_CompassNMD/10x/annotated_coords.csv')
df.head()
svs_path = df["svs_path"].values
svs_path = [os.path.basename(s).split(".svs")[0] for s in svs_path]

df["svs_name"] = svs_path
df.head()

In [ ]:
import os
import re


def parse_grid_search_params(path):
    folder_name = None

    for part in path.split(os.sep):
        if part.startswith("blending_mode_"):
            folder_name = part
            break

    if folder_name is None:
        raise ValueError(f"Cannot find grid-search folder in path: {path}")

    result = {
        "mode_overlap": "",
        "overlap": "",
        "sigma": "",
    }

    mode_match = re.search(r"blending_mode_([^_]+)", folder_name)
    overlap_match = re.search(r"overlap_([0-9]+p[0-9]+|[0-9]+)", folder_name)
    sigma_match = re.search(r"blending_sigma_([0-9]+p[0-9]+|[0-9]+)", folder_name)

    if mode_match:
        result["mode_overlap"] = mode_match.group(1)

    if overlap_match:
        result["overlap"] = float(overlap_match.group(1).replace("p", "."))

    if sigma_match:
        result["sigma"] = float(sigma_match.group(1).replace("p", "."))

    return result

path = "/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/TCGA_CompassNMD/Results/Histokit_30_06_2026/grid_search/blending_mode_constant__overlap_0p5/artifact_detection/grandqc/masks_cropped_color"

params = parse_grid_search_params(path)
print(params)

In [ ]:
main_dir = "/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/TCGA_CompassNMD/Results/Histokit_30_06_2026/grid_search"
folders_processed = os.listdir(main_dir)
folders_processed = [os.path.join(main_dir,f) for f in folders_processed if os.path.isdir(os.path.join(main_dir,f))]

In [ ]:

target_mag = 10
for folder in  tqdm(folders_processed, desc="Processing folders"):

    mask_dir = os.path.join(folder, "artifact_detection/grandqc/masks")
    saver = HDF5Saver()
    parsed_dir = os.path.join(folder, "artifact_detection/grandqc/masks_cropped_numeric")

    parsed_color_dir = os.path.join(folder, "artifact_detection/grandqc/masks_cropped_color")

    os.makedirs(parsed_color_dir, exist_ok=True)

    os.makedirs(parsed_dir, exist_ok=True)

    #print("Processing folder:", folder)
    for m in os.listdir(mask_dir):
            name = m.split(".h5")[0]
            rows = df[df["svs_name"] == name]

            for row in rows.itertuples(index=False):
                try:
                    save_name = row.patch_path.split("/")[-1].split(".tiff")[0]
                    out_path = os.path.join( parsed_color_dir, f"{save_name}.png")

                    if os.path.exists(out_path):
                        try:
                            with Image.open(out_path) as img:
                                img.verify()

                            with Image.open(out_path) as img:
                                img.load()

                            # print(f"Skipping existing valid file: {save_name}.png")
                            continue

                        except Exception as e:
                            print(f"Existing file is corrupted, regenerating: {out_path}")
                            print(e)

                    print(f"Processing {folder} file: {save_name}.png")


                    x10 = row.x_10x
                    y10 = row.y_10x
                    w10 = row.patch_width_10x
                    h10 = row.patch_height_10x

                    dict_mask = saver.load(os.path.join(mask_dir, m))
                    size = dict_mask['level_dimensions_0']
                    mag = dict_mask['mag_l0']
                    factor = target_mag / mag
                    size = np.round(size * factor).astype(int)
                    slide = np.zeros((size[1], size[0]), dtype=np.uint8)

                    for b, mask in zip(dict_mask["bbox"], dict_mask["mask"]):
                        x, y, w, h = [float(v) for v in b]

                        x0 = int(np.floor(x))
                        y0 = int(np.floor(y))
                        x1 = int(np.ceil(x + w))
                        y1 = int(np.ceil(y + h))

                        mask = mask.astype(np.uint8)

                        x0_clip = max(0, x0)
                        y0_clip = max(0, y0)
                        x1_clip = min(x1, slide.shape[1])
                        y1_clip = min(y1, slide.shape[0])

                        if x1_clip <= x0_clip or y1_clip <= y0_clip:
                            continue

                        roi = slide[y0_clip:y1_clip, x0_clip:x1_clip]

                        mask_x0 = x0_clip - x0
                        mask_y0 = y0_clip - y0

                        roi_h, roi_w = roi.shape[:2]

                        mask_crop = mask[
                            mask_y0:mask_y0 + roi_h,
                            mask_x0:mask_x0 + roi_w
                        ]

                        common_h = min(roi.shape[0], mask_crop.shape[0])
                        common_w = min(roi.shape[1], mask_crop.shape[1])

                        roi = roi[:common_h, :common_w]
                        mask_crop = mask_crop[:common_h, :common_w]

                        con = (roi == 0) & (mask_crop != 0)
                        roi[con] = mask_crop[con]

                        slide[
                            y0_clip:y0_clip + common_h,
                            x0_clip:x0_clip + common_w
                        ] = roi

                    slide = slide[round(int(y10)):round(int(y10+h10)), int(round(x10)):int(round(x10+w10))]
                    Image.fromarray(slide).save(os.path.join(parsed_dir, f"{save_name}.png"))

                    slide_color = np.zeros((slide.shape[0], slide.shape[1], 3), dtype=np.uint8)
                    slide_color[slide == 0] = [0, 0, 0]
                    slide_color[slide == 1] = [128, 128, 128]
                    slide_color[slide == 2] = [255, 99, 71]
                    slide_color[slide == 3] = [0, 255, 0]
                    slide_color[slide == 4] = [255, 0, 0]
                    slide_color[slide == 5] = [255, 0, 255]
                    slide_color[slide == 6] = [75, 0, 130]

                    Image.fromarray(slide_color).save(os.path.join(parsed_color_dir, f"{save_name}.png"))
                except Exception as e:
                    print(f"Error processing {m}")
                    print(e)




Sometimes we have overlaping tissue regions, so we need to mask this areas, to do that we used masks defined for ground truth.

In [ ]:
masks_dir = "/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/TCGA_CompassNMD/10x/annotated_masked_overlap/"
masks_names = os.listdir(masks_dir)

for folder in tqdm(folders_processed, desc="Processing folders"):
    parsed_color_dir = os.path.join(folder, "artifact_detection/grandqc/masks_cropped_color")

    for m_n in tqdm(masks_names, desc="Processing masks", leave=False):
        try:
            mask_path = os.path.join(masks_dir, m_n)
            pred_path = os.path.join(parsed_color_dir, m_n)

            mask = np.array(Image.open(mask_path).convert("L"))
            pred_mask = np.array(Image.open(pred_path).convert("RGB"))

            if mask.shape != pred_mask.shape[:2]:
                print(f"Shape mismatch for {m_n} in {folder}: mask {mask.shape}, pred {pred_mask.shape}")
                continue

            pred_mask[mask == 255] = [0, 0, 0]

            Image.fromarray(pred_mask).save(pred_path)

        except Exception as e:
            print(f"Error processing {m_n} in folder {folder}: {e}")





Create prediction masks - visualise segmentation results for multiclass classification and calculate statistics.

In [ ]:
from skimage.morphology import binary_opening, binary_closing, disk
from scipy.ndimage import binary_fill_holes
from PIL.ImageFile import ImageFile
from concurrent.futures import ThreadPoolExecutor, as_completed
from histokit.segmentation.evaluate.eval import evaluate_rgb_mask
ImageFile.LOAD_TRUNCATED_IMAGES = True
from skimage.morphology import remove_small_holes
import skimage
from PIL.ImageFile import ImageFile
from concurrent.futures import ThreadPoolExecutor, as_completed
from histokit.segmentation.evaluate.eval import evaluate_rgb_mask

gt_folder = "/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/TCGA_CompassNMD/10x/gt_mask"
masks = os.listdir(gt_folder)
res_binary_list = []
res_multiclass_list = []
errors = []
tasks = []

def process_single_mask(folder, mask, gt_folder, classes):
    masks_color = os.path.join(folder, "artifact_detection/grandqc/masks_cropped_color")
    vis = os.path.join(folder, "artifact_detection/grandqc/visualization_segmentation")
    processed = os.path.join(folder, "artifact_detection/grandqc/masks_cropped_color_postprocessed")
    os.makedirs(vis, exist_ok=True)
    os.makedirs(processed, exist_ok=True)
    params = parse_grid_search_params(folder)

    mask_basename = os.path.basename(mask)

    gt_path = os.path.join(gt_folder, mask)
    pred_path = os.path.join(masks_color, mask_basename)

    if not os.path.exists(pred_path):
        return None, None, f"Missing prediction for {mask_basename}"

    mask_gt = np.array(Image.open(gt_path).convert("RGB"))
    mask_pred = np.array(Image.open(pred_path).convert("RGB"))

    background = np.all(mask_pred == classes["Background"], axis=-1)
    selem = disk(3)
    edge = np.all(mask_pred == classes["Edge"], axis=-1)
    edge = skimage.morphology.opening(edge, footprint=selem)
    edge = skimage.morphology.closing(edge, footprint=selem)
    edge = binary_fill_holes(edge)
    mask_pred[edge] = classes["Edge"]

    tissue = np.all(mask_pred == classes["Tissue"], axis=-1)

    tissue_filled = remove_small_holes(
        tissue,
        max_size=int(0.001 * tissue.shape[0] * tissue.shape[1])
    )

    holes = tissue_filled & ~tissue
    mask_pred[holes] = classes["Tissue"]

    oof = np.all(mask_pred == classes["Out.Of.Focus"], axis=-1)
    bg = np.all(mask_pred == classes["Background"], axis=-1)

    oof_processed = skimage.morphology.opening(oof, footprint=selem)
    oof_processed = skimage.morphology.closing(oof_processed, footprint=selem)
    oof_processed = oof_processed & ~bg
    oof_processed = remove_small_holes(
        oof_processed,
        max_size=int(0.001 * tissue.shape[0] * tissue.shape[1])
    )

    mask_pred[oof_processed] = classes["Out.Of.Focus"]

    Image.fromarray(mask_pred).save(os.path.join(processed, mask_basename))

    if mask_gt.shape != mask_pred.shape:
        return None, None, (
            f"Shape mismatch for {mask_basename}: "
            f"GT {mask_gt.shape}, pred {mask_pred.shape}"
        )

    res_binary, res_multiclass = evaluate_rgb_mask(
        mask_gt=mask_gt,
        mask_pred=mask_pred,
        mask_basename=mask_basename,
        vis_dir=vis,
        method="HistoKit (no postprocessing)",
        tissue_class=[128, 128, 128],
        bg_class=[0, 0, 0],
        multiclass=True,
        class_dict=classes,
    )

    res_binary["Mode"] = params["mode_overlap"]
    res_binary["Overlap"] = params["overlap"]
    res_binary["Sigma"] = params["sigma"]
    res_multiclass["Mode"] = params["mode_overlap"]
    res_multiclass["Overlap"] = params["overlap"]
    res_multiclass["Sigma"] = params["sigma"]


    return res_binary, res_multiclass, None


res_binary_list = []
res_multiclass_list = []
errors = []
tasks = []

with ThreadPoolExecutor(max_workers=12) as executor:
    for folder in folders_processed:
        for mask in masks:
            tasks.append(
                executor.submit(
                    process_single_mask,
                    folder,
                    mask,
                    gt_folder,
                    classes,
                )
            )

    for future in tqdm(as_completed(tasks), total=len(tasks), desc="Processing masks"):
        try:
            res_binary, res_multiclass, error = future.result()

            if error is not None:
                errors.append(error)
                print(error)
                continue

            if res_binary is not None:
                res_binary_list.append(res_binary)

            if res_multiclass is not None:
                res_multiclass_list.append(res_multiclass)

        except Exception as e:
            errors.append(str(e))
            print(e)

df_binary = pd.DataFrame(res_binary_list)
df_multiclass = pd.DataFrame(res_multiclass_list)
df_errors = pd.DataFrame({"error": errors})

df_binary.to_csv("/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/TCGA_CompassNMD/Results/Histokit_30_06_2026/binary_metrics_processed.csv", index=False)
df_multiclass.to_csv("/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/TCGA_CompassNMD/Results/Histokit_30_06_2026/multiclass_metrics_processed.csv", index=False)
df_errors.to_csv("/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/TCGA_CompassNMD/Results/Histokit_30_06_2026/errors_processed.csv", index=False)



## Slidl Dataset

This dataset contains information about two classes: artifact free tissue and regions containing artifacts or background. Artifacts like out of focus regions are not annotated, so there is a need to exclude this type of artifact from the masks.

In [ ]:
main_dir = "/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/Slidl_dataset/Results/Histokit_30_06_2026/grid_search"
folders_processed = os.listdir(main_dir)
folders_processed = [os.path.join(main_dir,f) for f in folders_processed if os.path.isdir(os.path.join(main_dir,f))]


target_mag = 10
for folder in  tqdm(folders_processed, desc="Processing folders"):

    print("Processing folder:", folder)
    mask_dir = os.path.join(folder, "artifact_detection/grandqc/masks")
    saver = HDF5Saver()
    parsed_dir = os.path.join(folder, "artifact_detection/grandqc/masks_cropped_numeric")
    parsed_color_dir = os.path.join(folder, "artifact_detection/grandqc/masks_cropped_color")
    os.makedirs(parsed_color_dir, exist_ok=True)
    os.makedirs(parsed_dir, exist_ok=True)

    for m in os.listdir(mask_dir):
        try:
            name = m.split(".h5")[0]
            dict_mask = saver.load(os.path.join(mask_dir, m))
            size = dict_mask['level_dimensions_0']
            mag = dict_mask['mag_l0']
            factor = target_mag / mag
            size = np.round(size * factor).astype(int)
            slide = np.zeros((size[1], size[0]), dtype=np.uint8)

            for b, mask in zip(dict_mask["bbox"], dict_mask["mask"]):
                x, y, w, h = [float(v) for v in b]

                x0 = int(np.floor(x))
                y0 = int(np.floor(y))
                x1 = int(np.ceil(x + w))
                y1 = int(np.ceil(y + h))

                mask = mask.astype(np.uint8)

                x0_clip = max(0, x0)
                y0_clip = max(0, y0)
                x1_clip = min(x1, slide.shape[1])
                y1_clip = min(y1, slide.shape[0])

                if x1_clip <= x0_clip or y1_clip <= y0_clip:
                    continue

                roi = slide[y0_clip:y1_clip, x0_clip:x1_clip]

                mask_x0 = x0_clip - x0
                mask_y0 = y0_clip - y0

                roi_h, roi_w = roi.shape[:2]

                mask_crop = mask[
                    mask_y0:mask_y0 + roi_h,
                    mask_x0:mask_x0 + roi_w
                ]

                common_h = min(roi.shape[0], mask_crop.shape[0])
                common_w = min(roi.shape[1], mask_crop.shape[1])

                roi = roi[:common_h, :common_w]
                mask_crop = mask_crop[:common_h, :common_w]

                con = (roi == 0) & (mask_crop != 0)
                roi[con] = mask_crop[con]

                slide[
                    y0_clip:y0_clip + common_h,
                    x0_clip:x0_clip + common_w
                ] = roi

            Image.fromarray(slide).save(os.path.join(parsed_dir, f"{name}.png"))

            slide_color = np.zeros((slide.shape[0], slide.shape[1], 3), dtype=np.uint8)
            slide_color[slide == 0] = [0, 0, 0]
            slide_color[slide == 1] = [128, 128, 128]
            slide_color[slide == 2] = [255, 99, 71]
            slide_color[slide == 3] = [0, 255, 0]
            slide_color[slide == 4] = [255, 0, 0]
            slide_color[slide == 5] = [255, 0, 255]
            slide_color[slide == 6] = [75, 0, 130]

            Image.fromarray(slide_color).save(os.path.join(parsed_color_dir, f"{name}.png"))
        except Exception as e:
            print(f"Error processing {m}")
            print(e)

Check if all files exists both in gt and prediction folders.

In [ ]:
grid_folder = f"/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/Slidl_dataset/Results/Histokit_30_06_2026/grid_search/"
for pred_folder_main in tqdm(os.listdir(grid_folder), "Checking folders"):
    try:
        if not os.path.isdir(os.path.join(grid_folder, pred_folder_main)):
            continue
    except Exception as e:
        print(f"Error checking folder {pred_folder_main}")
        continue

    pred_folder = os.path.join(grid_folder, pred_folder_main, "artifact_detection/grandqc/masks_cropped_color")
    gt_dir = f"/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/Slidl_dataset/gt_masks/gt_mask"
    res = check_gt_pred_folders(gt_dir, pred_folder, use_ext = False)

    if not res:
        print(f"Mismatch in number of images in folder {pred_folder}")

In [ ]:
from skimage.morphology import remove_small_holes
import skimage
from PIL.ImageFile import ImageFile
from concurrent.futures import ThreadPoolExecutor, as_completed
from histokit.segmentation.evaluate.eval import evaluate_rgb_mask



main_dir = "/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/Slidl_dataset/Results/Histokit_30_06_2026/grid_search"
folders_processed = os.listdir(main_dir)
folders_processed = [os.path.join(main_dir,f) for f in folders_processed if os.path.isdir(os.path.join(main_dir,f))]

gt_folder = "/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/Slidl_dataset/gt_masks"
masks = os.listdir(gt_folder)
res_binary_list = []
res_multiclass_list = []
errors = []
tasks = []

classes = {
    "Tissue": [255, 255, 255],
    "Other": [0, 0, 0],
}


def process_single_mask(folder, mask, gt_folder, classes):


    masks_color = os.path.join(folder, "artifact_detection/grandqc/masks_cropped_color")
    vis = os.path.join(folder, "artifact_detection/grandqc/visualization_segmentation")
    postprocessed = os.path.join(folder, "artifact_detection/grandqc/masks_cropped_color_postprocessed")
    os.makedirs(vis, exist_ok=True)
    os.makedirs(postprocessed, exist_ok=True)
    params = parse_grid_search_params(folder)

    mask_basename = os.path.basename(mask)

    gt_path = os.path.join(gt_folder, mask)
    pred_path = os.path.join(masks_color, mask_basename)

    if not os.path.exists(pred_path):
        return None, None, f"Missing prediction for {mask_basename}"

    mask_gt = np.array(Image.open(gt_path).convert("RGB"))

    # GT: white = tissue, everything else = background
    gt_tissue = np.all(mask_gt == [255, 255, 255], axis=-1)
    mask_gt_bin = np.zeros_like(mask_gt, dtype=np.uint8)
    mask_gt_bin[gt_tissue] = [255, 255, 255]

    mask_pred = np.array(Image.open(pred_path).convert("RGB"))

    # Prediction: tissue + Out.Of.Focus = tissue, everything else = background
    pred_tissue = (
        np.all(mask_pred == [128, 128, 128], axis=-1) |
        np.all(mask_pred == [75, 0, 130], axis=-1)
    )

    mask_pred_bin = mask_gt_bin
    mask_pred_bin[pred_tissue] = [255, 255, 255]

    mask_bool = mask_pred_bin[:, :, 0] > 0

    mask_pred_bin_filled = remove_small_holes(
        mask_bool,
        max_size=int(0.001 * mask_bool.shape[0] * mask_bool.shape[1]),
    )

    mask_pred_bin_filled = (mask_pred_bin_filled.astype(np.uint8) * 255)

    Image.fromarray(mask_pred_bin_filled).save(
        os.path.join(postprocessed, mask_basename)
    )


    mask_gt = np.stack([mask_pred_bin_filled] * 3, axis=-1)
    mask_pred = np.stack([mask_pred_bin_filled] * 3, axis=-1)

    mask_gt_img = Image.fromarray(mask_gt)
    mask_pred_img = Image.fromarray(mask_pred)

    if mask_gt_img.size != mask_pred_img.size:
        print(
            f"Resizing {mask_basename}: "
            f"pred {mask_pred_img.size} -> GT {mask_gt_img.size}"
        )

        mask_pred = np.array(mask_pred_img.resize(
            mask_gt_img.size,
            resample=Image.Resampling.NEAREST,
        ))

    res_binary, res_multiclass = evaluate_rgb_mask(
        mask_gt=mask_gt,
        mask_pred=mask_pred,
        mask_basename=mask_basename,
        vis_dir=vis,
        method="HistoKit (no postprocessing)",
        tissue_class=[255, 255, 255],
        bg_class=[0, 0, 0],
        multiclass=True,
        class_dict=classes,
    )

    res_binary["Mode"] = params["mode_overlap"]
    res_binary["Overlap"] = params["overlap"]
    res_binary["Sigma"] = params["sigma"]
    res_multiclass["Mode"] = params["mode_overlap"]
    res_multiclass["Overlap"] = params["overlap"]
    res_multiclass["Sigma"] = params["sigma"]


    return res_binary, res_multiclass, None


res_binary_list = []
res_multiclass_list = []
errors = []
tasks = []

with ThreadPoolExecutor(max_workers=2) as executor:
    for folder in folders_processed:
        for mask in masks:
            tasks.append(
                executor.submit(
                    process_single_mask,
                    folder,
                    mask,
                    gt_folder,
                    classes,
                )
            )

    for future in tqdm(as_completed(tasks), total=len(tasks), desc="Processing masks"):
        try:
            res_binary, res_multiclass, error = future.result()

            if error is not None:
                errors.append(error)
                print(error)
                continue

            if res_binary is not None:
                res_binary_list.append(res_binary)

            if res_multiclass is not None:
                res_multiclass_list.append(res_multiclass)

        except Exception as e:
            errors.append(str(e))
            print(e)

df_binary = pd.DataFrame(res_binary_list)
df_multiclass = pd.DataFrame(res_multiclass_list)
df_errors = pd.DataFrame({"error": errors})

df_binary.to_csv("/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/Slidl_dataset/Results/Histokit_30_06_2026/binary_metrics.csv", index=False)
df_multiclass.to_csv("/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/Slidl_dataset/Results/Histokit_30_06_2026/multiclass_metrics.csv", index=False)
df_errors.to_csv("/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/Slidl_dataset/Results/Histokit_30_06_2026/errors.csv", index=False)

## GrandQC Test Dataset

In this dataset, we exclude background prediction, due to the fact that patches contain tissue and annotated artifacts.

In [ ]:
organs = ["Breast"]
target_mag = 10
for o in organs:
    main_dir = f"/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/GrandQC Test Dataset/PreprocessedDataset/{o}/10x/Results/Histokit_30_06_2026/grid_search"
    dirs_img = os.listdir(main_dir)
    folders_processed = [os.path.join(main_dir,f) for f in dirs_img]

    for folder in  tqdm(folders_processed, desc="Processing folders"):

        print("Processing folder:", folder)
        mask_dir = os.path.join(folder, "artifact_detection/grandqc/masks")
        saver = HDF5Saver()
        parsed_dir = os.path.join(folder, "artifact_detection/grandqc/masks_cropped_numeric")
        parsed_color_dir = os.path.join(folder, "artifact_detection/grandqc/masks_cropped_color")
        os.makedirs(parsed_color_dir, exist_ok=True)
        os.makedirs(parsed_dir, exist_ok=True)

        for m in os.listdir(mask_dir):
            try:
                name = m.split(".h5")[0]
                dict_mask = saver.load(os.path.join(mask_dir, m))
                size = dict_mask['level_dimensions_0']
                mag = dict_mask['mag_l0']
                factor = target_mag / mag
                size = np.round(size * factor).astype(int)
                slide = np.zeros((size[1], size[0]), dtype=np.uint8)

                for b, mask in zip(dict_mask["bbox"], dict_mask["mask"]):
                    x, y, w, h = [float(v) for v in b]

                    x0 = int(np.floor(x))
                    y0 = int(np.floor(y))
                    x1 = int(np.ceil(x + w))
                    y1 = int(np.ceil(y + h))

                    mask = mask.astype(np.uint8)

                    x0_clip = max(0, x0)
                    y0_clip = max(0, y0)
                    x1_clip = min(x1, slide.shape[1])
                    y1_clip = min(y1, slide.shape[0])

                    if x1_clip <= x0_clip or y1_clip <= y0_clip:
                        continue

                    roi = slide[y0_clip:y1_clip, x0_clip:x1_clip]

                    mask_x0 = x0_clip - x0
                    mask_y0 = y0_clip - y0

                    roi_h, roi_w = roi.shape[:2]

                    mask_crop = mask[
                        mask_y0:mask_y0 + roi_h,
                        mask_x0:mask_x0 + roi_w
                    ]

                    common_h = min(roi.shape[0], mask_crop.shape[0])
                    common_w = min(roi.shape[1], mask_crop.shape[1])

                    roi = roi[:common_h, :common_w]
                    mask_crop = mask_crop[:common_h, :common_w]

                    con = (roi == 0) & (mask_crop != 0)
                    roi[con] = mask_crop[con]

                    slide[
                        y0_clip:y0_clip + common_h,
                        x0_clip:x0_clip + common_w
                    ] = roi

                Image.fromarray(slide).save(os.path.join(parsed_dir, f"{name}.png"))

                slide_color = np.zeros((slide.shape[0], slide.shape[1], 3), dtype=np.uint8)
                slide_color[slide == 0] = [0, 0, 0]
                slide_color[slide == 1] = [128, 128, 128]
                slide_color[slide == 2] = [255, 99, 71]
                slide_color[slide == 3] = [0, 255, 0]
                slide_color[slide == 4] = [255, 0, 0]
                slide_color[slide == 5] = [255, 0, 255]
                slide_color[slide == 6] = [75, 0, 130]

                Image.fromarray(slide_color).save(os.path.join(parsed_color_dir, f"{name}.png"))
            except Exception as e:
                print(f"Error processing {m}")
                print(e)


In [ ]:
## Check number of images in each folder (should be the same as the number of gt images)
organs = ["Kidney","Colon", "Breast"]

for o in organs:
    grid_folder = f"/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/GrandQC Test Dataset/PreprocessedDataset/{o}/10x/Results/Histokit_30_06_2026/grid_search"
    for pred_folder_main in tqdm(os.listdir(grid_folder), "Checking folders for organ: " + o):
        try:
            if not os.path.isdir(os.path.join(grid_folder, pred_folder_main)):
                continue
        except Exception as e:
            print(f"Error checking folder {pred_folder_main} for organ {o}: {e}")
            continue

        pred_folder = os.path.join(grid_folder, pred_folder_main, "artifact_detection/grandqc/masks_cropped_color")
        gt_dir = f"/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/GrandQC Test Dataset/PreprocessedDataset/{o}/10x/gt_mask"
        res = check_gt_pred_folders(gt_dir, pred_folder, use_ext = False)

        for pred_img_pth in os.listdir(pred_folder):
            pred_img = Image.open(os.path.join(pred_folder, pred_img_pth))
            gt_img = Image.open(os.path.join(gt_dir, pred_img_pth))
            if pred_img.size != gt_img.size:
                print(f"Size mismatch for {pred_img} in organ {o}: GT {gt_img.size}, pred {pred_img.size}")

        if not res:
            print(f"Mismatch in number of images for organ {o} in folder {pred_folder}")

In [ ]:
from histokit.slide import Slide

s = Slide("/mnt/warehouse/Projects/HE/Data/Artifacts Segmentation/Slidl_dataset/svs/tumor_021.tif")